# Custom Churro Run Metrics Analysis

This notebook finds run folders named:
- `results/custom_churro_infer_run<N>`
- `results/custom_churro_infer_run_run<N>`

Then it reads `vllm/test/all_metrics.json` from each run and creates:
1. Per-run score summary table (`printed`, `handwritten`, `overall`)
2. Bar chart of run averages
3. Language heatmap for printed documents
4. Language heatmap for handwritten documents


In [2]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_DIR_PATTERN = re.compile(r"^custom_churro_infer_run(?:_run)?(\d+)$")
METRICS_RELATIVE_PATH = Path("vllm") / "test" / "all_metrics.json"
MAIN_SCORE_KEY = "normalized_levenshtein_similarity"

DOC_TYPE_ALIASES = {
    "print": "print",
    "printed": "print",
    "handwriting": "handwriting",
    "handwritten": "handwriting",
}


def normalize_doc_type(value):
    if value is None:
        return None
    return DOC_TYPE_ALIASES.get(str(value).strip().lower())


def to_float(value):
    return float(value) if isinstance(value, (int, float)) else None


def discover_metrics_files(search_roots):
    metrics_files = []
    for root in search_roots:
        root = Path(root)
        if not root.exists() or not root.is_dir():
            continue

        # search_roots are expected to point at the concrete results directory
        # (e.g. /scratch/project_2017385/Churro_copy/results).
        for run_dir in root.glob("custom_churro_infer_run*"):
            if not run_dir.is_dir():
                continue
            if not RUN_DIR_PATTERN.match(run_dir.name):
                continue
            metrics_path = run_dir / METRICS_RELATIVE_PATH
            if metrics_path.is_file():
                metrics_files.append(metrics_path)

    return sorted(set(metrics_files), key=lambda p: str(p))


def parse_language_type_metrics(raw_metrics):
    print_scores = {}
    handwriting_scores = {}

    if not isinstance(raw_metrics, dict):
        return print_scores, handwriting_scores

    has_nested_buckets = False
    for key, value in raw_metrics.items():
        doc_type = normalize_doc_type(key)
        if doc_type in ("print", "handwriting") and isinstance(value, dict):
            has_nested_buckets = True
            break

    if has_nested_buckets:
        for bucket_name, language_map in raw_metrics.items():
            doc_type = normalize_doc_type(bucket_name)
            if doc_type not in ("print", "handwriting"):
                continue
            if not isinstance(language_map, dict):
                continue
            target = print_scores if doc_type == "print" else handwriting_scores
            for language, score in language_map.items():
                score_float = to_float(score)
                if score_float is not None:
                    target[str(language)] = score_float
        return print_scores, handwriting_scores

    for key, score in raw_metrics.items():
        key = str(key)
        if "_" not in key:
            continue
        language, raw_doc_type = key.rsplit("_", 1)
        doc_type = normalize_doc_type(raw_doc_type)
        score_float = to_float(score)
        if doc_type == "print" and score_float is not None:
            print_scores[language] = score_float
        elif doc_type == "handwriting" and score_float is not None:
            handwriting_scores[language] = score_float

    return print_scores, handwriting_scores


def parse_run_metrics(metrics_path):
    run_dir = metrics_path.parent.parent.parent
    run_name = run_dir.name
    match = RUN_DIR_PATTERN.match(run_name)
    run_index = int(match.group(1)) if match else -1
    variant = "run_run" if "_run_run" in run_name else "run"

    if run_dir.parent.name == "results":
        source_name = run_dir.parent.parent.name
    else:
        source_name = run_dir.parent.name

    with open(str(metrics_path), "r") as f:
        metrics = json.load(f)

    type_metrics = metrics.get("type_metrics", {})
    aggregate_metrics = metrics.get("aggregate_metrics", {})
    language_and_type = metrics.get("main_language_and_type_metrics", {})

    printed_score = None
    handwritten_score = None
    if isinstance(type_metrics, dict):
        for key, value in type_metrics.items():
            doc_type = normalize_doc_type(key)
            score = to_float(value)
            if doc_type == "print":
                printed_score = score
            elif doc_type == "handwriting":
                handwritten_score = score

    overall_score = None
    if isinstance(aggregate_metrics, dict):
        overall_score = to_float(aggregate_metrics.get(MAIN_SCORE_KEY))

    print_language_scores, handwriting_language_scores = parse_language_type_metrics(language_and_type)

    return {
        "source_name": source_name,
        "run_name": run_name,
        "run_index": run_index,
        "variant": variant,
        "run_path": run_dir,
        "metrics_path": metrics_path,
        "printed_score": printed_score,
        "handwritten_score": handwritten_score,
        "overall_score": overall_score,
        "print_language_scores": print_language_scores,
        "handwriting_language_scores": handwriting_language_scores,
        "run_label": "",
    }


def load_all_runs(search_roots):
    metrics_files = discover_metrics_files(search_roots)
    runs = []
    for metrics_path in metrics_files:
        try:
            runs.append(parse_run_metrics(metrics_path))
        except Exception as exc:
            print("[skip] Could not parse {}: {}".format(metrics_path, exc))

    name_counts = {}
    for run in runs:
        name_counts[run["run_name"]] = name_counts.get(run["run_name"], 0) + 1

    for run in runs:
        if name_counts.get(run["run_name"], 0) > 1:
            run["run_label"] = "{}/{}".format(run["source_name"], run["run_name"])
        else:
            run["run_label"] = run["run_name"]

    runs.sort(
        key=lambda r: (
            r["run_index"],
            0 if r["variant"] == "run" else 1,
            r["source_name"],
            r["run_name"],
        )
    )
    return runs


def build_summary_dataframe(runs):
    rows = []
    for run in runs:
        rows.append(
            {
                "run_label": run["run_label"],
                "source": run["source_name"],
                "run_name": run["run_name"],
                "run_index": run["run_index"],
                "printed_score": run["printed_score"],
                "handwritten_score": run["handwritten_score"],
                "overall_score": run["overall_score"],
                "metrics_path": str(run["metrics_path"]),
            }
        )

    summary_df = pd.DataFrame(rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            by=["run_index", "run_name", "source"],
            ascending=[True, True, True],
        ).reset_index(drop=True)
    return summary_df


def build_language_dataframe(runs, doc_type):
    if doc_type not in ("print", "handwriting"):
        raise ValueError("doc_type must be either 'print' or 'handwriting'")

    ordered_runs = sorted(
        runs,
        key=lambda r: (r.get("run_index", -1), r.get("run_name", "")),
    )

    language_maps = []
    for run in ordered_runs:
        if doc_type == "print":
            language_maps.append(run["print_language_scores"])
        else:
            language_maps.append(run["handwriting_language_scores"])

    languages = sorted(
        set(
            language
            for language_map in language_maps
            for language in language_map.keys()
        )
    )

    if not languages:
        return pd.DataFrame(index=[str(run.get("run_index", "")) for run in ordered_runs])

    matrix_rows = []
    for run in ordered_runs:
        if doc_type == "print":
            language_map = run["print_language_scores"]
        else:
            language_map = run["handwriting_language_scores"]

        row = {language: language_map.get(language, np.nan) for language in languages}
        row["run_number"] = str(run.get("run_index", ""))
        matrix_rows.append(row)

    df = pd.DataFrame(matrix_rows).set_index("run_number")
    return df


def plot_run_averages(summary_df):
    metric_columns = {
        "printed_score": "Printed",
        "handwritten_score": "Handwritten",
        "overall_score": "Overall",
    }

    ordered = summary_df.sort_values(by=["run_index", "run_name"]).copy()
    ordered["run_number"] = ordered["run_index"].astype(int).astype(str)

    plot_df = ordered[["run_number", "printed_score", "handwritten_score", "overall_score"]]
    plot_df = plot_df.set_index("run_number")[["printed_score", "handwritten_score", "overall_score"]]
    plot_df = plot_df.rename(columns=metric_columns).astype(float)

    fig_width = max(10, len(plot_df) * 1.3)
    fig, ax = plt.subplots(figsize=(fig_width, 6))
    plot_df.plot(
        kind="bar",
        ax=ax,
        width=0.8,
        color=["#1f77b4", "#ff7f0e", "#2ca02c"],
    )

    ax.set_title("Average Scores Per Run", fontsize=14)
    ax.set_ylabel("Score (%)")
    ax.set_xlabel("Run Number")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Metric", frameon=True)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")

    for patch in ax.patches:
        height = patch.get_height()
        if np.isfinite(height):
            ax.annotate(
                "{:.1f}".format(height),
                (patch.get_x() + patch.get_width() / 2.0, height),
                ha="center",
                va="bottom",
                fontsize=8,
                xytext=(0, 2),
                textcoords="offset points",
            )

    fig.tight_layout()
    return fig, ax


def plot_language_heatmap(language_df, title):
    if language_df.empty or language_df.shape[1] == 0:
        print("[info] Skipping heatmap: no data for {}".format(title))
        return None, None

    values = language_df.values.astype(float)
    masked = np.ma.masked_invalid(values)

    cmap = plt.get_cmap("YlGnBu")
    if hasattr(cmap, "copy"):
        cmap = cmap.copy()
    if hasattr(cmap, "set_bad"):
        cmap.set_bad(color="#f0f0f0")

    vmin = 0.0
    vmax = 100.0
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    fig_width = max(10, language_df.shape[1] * 0.65 + 4)
    fig_height = max(4, language_df.shape[0] * 0.55 + 2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    image = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    colorbar = fig.colorbar(image, ax=ax, pad=0.02)
    colorbar.set_label("Score (%)")

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Language")
    ax.set_ylabel("Run Number")
    ax.set_xticks(np.arange(language_df.shape[1]))
    ax.set_xticklabels(language_df.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(language_df.shape[0]))
    ax.set_yticklabels(language_df.index)

    for i in range(language_df.shape[0]):
        for j in range(language_df.shape[1]):
            score = language_df.iat[i, j]
            if pd.notna(score):
                rgba = cmap(norm(float(score)))
                luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
                text_color = "white" if luminance < 0.5 else "black"
                ax.text(
                    j,
                    i,
                    "{:.1f}".format(score),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=text_color,
                )

    fig.tight_layout()
    return fig, ax


def maybe_save_figure(fig, output_dir, file_name):
    if fig is None:
        return
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / file_name
    fig.savefig(str(output_path), dpi=220, bbox_inches="tight")
    print("[saved] {}".format(output_path))


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Configuration
# Restrict analysis strictly to this directory:
# /scratch/project_2017385/Churro_copy/results/churro_original_prompt
RESULTS_ROOT = Path('/scratch/project_2017385/Churro_copy/results/churro_original_prompt')
SEARCH_ROOTS = [RESULTS_ROOT]

print('Working directory:', Path.cwd())
print('Results root:', RESULTS_ROOT)
print('Results root exists:', RESULTS_ROOT.exists())

# Always save figures for download in the same results directory.
SAVE_FIGURES = True
FIGURE_OUTPUT_DIR = RESULTS_ROOT
print('Figures will be saved to:', FIGURE_OUTPUT_DIR)


In [ ]:
runs = load_all_runs(SEARCH_ROOTS)

if not runs:
    print('No metrics discovered. Quick diagnostic of run directories:')
    discovered_dirs = []
    for root in SEARCH_ROOTS:
        discovered_dirs.extend(sorted([p for p in root.rglob('custom_churro_infer_run*') if p.is_dir()], key=str))

    if not discovered_dirs:
        print(' - No matching run directories found under configured roots.')
    else:
        for d in discovered_dirs:
            metrics_file = d / 'vllm' / 'test' / 'all_metrics.json'
            status = 'OK' if metrics_file.exists() else 'MISSING all_metrics.json'
            print(' - {} -> {}'.format(d, status))

    print('No valid metrics files were found. Skipping summary and plots.')
    summary_df = pd.DataFrame()
else:
    summary_df = build_summary_dataframe(runs)
    print('Loaded {} runs.'.format(len(summary_df)))

    summary_columns = [
        'run_label',
        'printed_score',
        'handwritten_score',
        'overall_score',
        'metrics_path',
    ]

    print('')
    print('Per-run score summary:')
    print(summary_df[summary_columns].to_string(index=False))

    overall_mean = summary_df[['printed_score', 'handwritten_score', 'overall_score']].mean(numeric_only=True)
    overall_mean = overall_mean.rename({'printed_score': 'printed', 'handwritten_score': 'handwritten'})
    print('')
    print('Mean scores across all discovered runs:')
    print(overall_mean.to_string())


In [ ]:
if summary_df.empty:
    print('No summary data available, so plots were skipped.')
else:
    fig1, _ = plot_run_averages(summary_df)
    if SAVE_FIGURES:
        maybe_save_figure(fig1, FIGURE_OUTPUT_DIR, 'run_average_scores.png')

    print_lang_df = build_language_dataframe(runs, 'print')
    handwriting_lang_df = build_language_dataframe(runs, 'handwriting')

    fig2, _ = plot_language_heatmap(print_lang_df, 'Printed Document Scores by Language and Run')
    if SAVE_FIGURES:
        maybe_save_figure(fig2, FIGURE_OUTPUT_DIR, 'language_heatmap_printed.png')

    fig3, _ = plot_language_heatmap(
        handwriting_lang_df,
        'Handwritten Document Scores by Language and Run',
    )
    if SAVE_FIGURES:
        maybe_save_figure(fig3, FIGURE_OUTPUT_DIR, 'language_heatmap_handwritten.png')

    plt.show()